In [ ]:
import numpy as np
import numpy.ma as ma #use masked array
import xarray as xr
import netCDF4 as nc #load and write netcdf data
from datetime import date, timedelta, datetime #create file history with creation date
from tqdm import tqdm #create a user-friendly feedback while script is running
from os import listdir
from os.path import isfile, join
import os
import re #Use RegEx 
import pandas as pd #handle dataframes
import cc3d #connected components patterns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as clrs
from scipy import stats
from scipy import signal
import json
import glob
import dask
from Code.CORDEX_EUR11.utils import *
#from dask.distributed import Client

SyntaxError: invalid syntax (2790966720.py, line 24)

In [6]:
os.chdir("/home/user/These/cordex_htws_cc3d")

In [7]:
read_directory_historical = "Data/historical"
read_directory_rcp = "Data/rcp85"
write_directory = "Data/output"

In [10]:
start_year = 1975
end_year = 2099
start_year_ref = 1975
end_year_ref = 2025
temp_variable = 'tasmax'
threshold_value = 95
distrib_window_size = 15
anomaly = True
relative_threshold = True
nb_days = 4
connectivity = 26

split_year=2005

year = 1975

In [12]:
def create_files_list_to_load(read_directory_historical=None,read_directory_rcp=None,start_year=1975,end_year=split_year):
    """Creates the list of files to load, and checks for the time consistency of the created list."""
    # Create sorted list of all files in directory
    if read_directory_rcp == None and read_directory_historical == None:
        raise ValueError("Both read_directory_rcp and read_directory_historical are None. At least one must be defined")
    elif read_directory_historical == None:
        existing_files = np.sort(glob.glob(join(read_directory_rcp,"*.nc")))
    elif read_directory_rcp == None:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc")))
    else:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc"))+glob.glob(join(read_directory_rcp,"*.nc")))
    # Select files matching the study period
    files_to_load = []
    for file in existing_files:
        file_start_year = int(file[-20:-16])
        file_end_year = int(file[-11:-7])
        if (file_start_year >= start_year and file_start_year <= end_year) or (file_end_year >= start_year and file_end_year <= end_year):
            files_to_load.append(str(file))
    # Check time consistency
    if len(files_to_load)==0:
        raise ValueError("files_to_load is empty. Check directories and time boundaries.")

    first_file_start_year = int(files_to_load[0][-20:-16])
    if start_year < first_file_start_year :
        raise ValueError(f"Start year {start_year} is before beginning of first file {first_file_start_year}.")
    last_file_end_year = int(files_to_load[-1][-11:-7])
    if end_year > last_file_end_year :
        raise ValueError(f"End year {end_year} is after ending of last file {last_file_end_year}.")

    for i in range(1,len(files_to_load)):
        previous_file_end_year = int(files_to_load[i-1][-11:-7])
        file_start_year = int(files_to_load[i][-20:-16])
        if file_start_year != previous_file_end_year+1 :
            raise ValueError(f"Time is not consistent in files_to_load. Missing year(s) between {previous_file_end_year} and {file_start_year}.")
    print(f"Time is consistent between start year {start_year} and end year {end_year}.")
    return(files_to_load)

In [13]:
# Load dictionary holding the time boundaries of each RWL
with open(join(write_directory,'regional_warming_levels.json'), 'r') as f:
    RWL_dict = json.load(f)

# Get grid_mapping
grid_mapping = xr.open_dataset(join(write_directory,f"labels_cc3d_year_{start_year}_ref_{start_year_ref}_{end_year_ref}.nc"), engine="netcdf4").label.grid_mapping

# Create list of temperature files to load
if end_year <= split_year: # If all files are in historical directory, ignore rcp directory
    files_to_load = create_files_list_to_load(read_directory_historical=read_directory_historical,read_directory_rcp=None,start_year=start_year,end_year=end_year)
elif start_year > split_year: # If all files are in rcp directory, ignore historical directory
    files_to_load = create_files_list_to_load(read_directory_historical=None,read_directory_rcp=read_directory_rcp,start_year=start_year,end_year=end_year)
else:
    files_to_load = create_files_list_to_load(read_directory_historical,read_directory_rcp,start_year,end_year)

da_threshold = xr.open_dataarray(join(write_directory,f"distrib_threshold_{threshold_value}.nc"),engine='netcdf4')
mask = (da_threshold.dayofyear>=152) & (da_threshold.dayofyear<=243) # dayofyear ranges from 1 to 365 ; 152 is June 1st, 243 is August 31st
da_threshold = da_threshold.sel(dayofyear=mask)

if anomaly :
    seasonal_cycle = xr.open_dataarray(join(write_directory,f"seasonal_cycle_{start_year_ref}_{end_year_ref}.nc"), engine='netcdf4')
    seasonal_cycle = seasonal_cycle.sel(dayofyear=mask) # Keep only JJA values

da_HWMId = xr.open_dataarray(join(write_directory,"Russo_HWMId.nc"),engine='netcdf4')

# Load population files
# FPOP files
ssp_file_dict = {}
for ssp in range(1,6):
    #ssp_file_dict[f"ds_pop_ssp{ssp}"] = xr.open_dataset(join(write_directory,"../..","FPOP",f"FPOP_SSP{ssp}_2020_2100_{grid_mapping}.nc"),engine='netcdf4')
    ssp_file_dict[f"ds_pop_ssp{ssp}"] = xr.open_dataset(join(write_directory,"../..","Data","FPOP",f"FPOP_SSP{ssp}_2020_2100_{grid_mapping}.nc"),engine='netcdf4')
# GHS-POP file
#ds_ghs_pop = xr.open_dataset(join(write_directory,"../..","GHS-POP",f"GHS_POP_1975_2030_{grid_mapping}.nc"),engine='netcdf4')
ds_ghs_pop = xr.open_dataset(join(write_directory,"../..","Data","GHS-POP",f"GHS_POP_1975_2030_{grid_mapping}.nc"),engine='netcdf4')

# Load cell area
#ds_cell_area = xr.open_dataset(join("/data/tmandonnet/CORDEX","cellarea",f"gridarea_CORDEX_EUR11_{grid_mapping}.nc"),engine='netcdf4') # Area of each grid cell, in m²
ds_cell_area = xr.open_dataset(join("/home/user/These/cordex_htws_cc3d/Data","cellarea",f"gridarea_CORDEX_EUR11_{grid_mapping}.nc"),engine='netcdf4') # Area of each grid cell, in m²
da_cell_area = ds_cell_area.cell_area/1e6 # Load DataArray and convert to km²

# Create DataFrame
df_htws = pd.DataFrame(data=None,columns=['Year','Start Date','End Date','RWL_1','RWL_2','RWL_3','RWL_4', # Account for the fact that RWL can overlap. Create 4 RWL columns to avoid edge cases but RWL_3 and RWL_4 will very likely remain empty
'Intensity','Spatial extent','Duration','Max','HWMId_sum',
'Exposed_population_ghs','HWMId_pop_ghs','Exposed_population_ssp1','HWMId_pop_ssp1',
'Exposed_population_ssp2','HWMId_pop_ssp2','Exposed_population_ssp3','HWMId_pop_ssp3',
'Exposed_population_ssp4','HWMId_pop_ssp4','Exposed_population_ssp5','HWMId_pop_ssp5']) # Create DataFrame

# Initialize variables used to find the correct temperature file to load for each year
loaded_temp_file = None
old_loaded_temp_file = None

# Compute weights for latitude-weighted mean
weights = np.cos(np.deg2rad(da_HWMId.lat))
weights.name = "weights"


Time is consistent between start year 1975 and end year 2099.


In [21]:
ds_temp = xr.open_dataset(join(read_directory_historical,"tasmax_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v1_day_19710101-19751231.nc"),engine='netcdf4')
#ds_temp = ds_temp.convert_calendar("noleap") # Drop 29th Feb or convert 360-day to 365-day
da_temp = getattr(ds_temp, temp_variable)
da_temp = da_temp.sel(time=(da_temp.time.dt.year==year)) # Keep only correct year
da_temp = da_temp.sel(time=(da_temp.time.dt.season=='JJA')) # Keep only JJA
if anomaly:
    da_temp = da_temp - seasonal_cycle.data # Compute anomaly
da_temp = da_temp - da_threshold.data # Compute threshold exceedance

In [22]:
ds_labels = xr.open_dataset(join(write_directory,f"labels_cc3d_year_{year}_ref_{start_year_ref}_{end_year_ref}.nc"), engine="netcdf4") # Load the corresponding nc file
da_labels = ds_labels.label

# Select correct year for HWMId and keep only JJA
da_HWMId_year = da_HWMId.sel(time=(da_HWMId.time.dt.year==year))
da_HWMId_year = da_HWMId_year.sel(time=(da_HWMId_year.time.dt.season=='JJA'))

# Create list of labels to iterate over
labels_list = np.unique(da_labels.data)
labels_list = labels_list[np.where(labels_list!=0)] # Remove 0 which is not a heatwave label


In [23]:
label = labels_list[0]
print(label)

24


In [33]:
df_htws.loc[label,"Year"]=year
# Find the corresponding RWL(s)
rwl_count=0
for rwl in RWL_dict :
    if (RWL_dict[rwl] != None) and (year in range(RWL_dict[rwl]["start_year"],RWL_dict[rwl]["end_year"]+1)):
        rwl_count += 1 # Find the correct column to fill
        df_htws.loc[label,f"RWL_{rwl_count}"] = float(rwl) # Record the corresponding RWL
da = da_labels
#TODO Check this entire block
da_bool_htw = da.where(da==label, drop=True).fillna(0)>0 # Select days and grid points for the heatwave of interest and convert to bool array
# Workaround bug in da_temp.where, could not find the explanation of the behaviour
# da_temp_htw = da_temp.where(da==label, drop=True)
temp_copy = da.copy()
temp_copy.data = da_temp.data
da_temp_htw = temp_copy.where(da==label, drop=True)
da_HWMId_htw = da_HWMId_year.where(da==label, drop=True)

In [34]:
df_htws.loc[label,'Year'] = year
df_htws.loc[label,'Start Date'] = da_temp_htw.time.data[0]
df_htws.loc[label,'End Date'] = da_temp_htw.time.data[-1]

In [37]:
df_htws

,Year,Start Date,End Date,RWL_1,RWL_2,RWL_3,RWL_4,Intensity,Spatial extent,Duration,...,Exposed_population_ssp1,HWMId_pop_ssp1,Exposed_population_ssp2,HWMId_pop_ssp2,Exposed_population_ssp3,HWMId_pop_ssp3,Exposed_population_ssp4,HWMId_pop_ssp4,Exposed_population_ssp5,HWMId_pop_ssp5
24,1975,1975-06-01 12:00:00,1975-06-11 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
da_label = xr.open_dataarray(join(write_directory,"labels_cc3d_year_1975_ref_1975_2025.nc"),engine='netcdf4')

In [18]:
labels_list = np.unique(da_label.data)[1:]
label = labels_list[1]

In [19]:
da_HWMId_year

<xarray.DataArray 'HWMId' (time: 92, rlat: 412, rlon: 424)> Size: 64MB
[16071296 values with dtype=float32]
Coordinates:
  * time     (time) object 736B 1975-06-01 12:00:00 ... 1975-08-31 12:00:00
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB ...
    lon      (rlat, rlon) float32 699kB ...
    height   float64 8B ...
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [161]:
da_HWMId_year.rlat

<xarray.DataArray 'rlat' (rlat: 412)> Size: 3kB
array([-23.375, -23.265, -23.155, ...,  21.615,  21.725,  21.835], shape=(412,))
Coordinates:
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
    height   float64 8B ...
Attributes:
    units:          degrees
    axis:           Y
    long_name:      latitude in rotated pole grid
    standard_name:  grid_latitude

In [29]:
da_temp.data == da_HWMId_year

<xarray.DataArray 'HWMId' (time: 92, rlat: 412, rlon: 424)> Size: 16MB
array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
...
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]]],
      shape=(92, 412, 424))
Coordinates:
  * time     (time) object 736B 1975-06-01 12:00:00 ... 1975-08-31 12:00:00
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB ...
    lon      (rlat, rlon) float32 699kB ...
    height   float64 8B ...
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [159]:
np.shape(da_temp)

(92, 412, 424)

In [20]:
da = da_label
#TODO Check this entire block
da_bool_htw = da.where(da==label, drop=True).fillna(0)>0 # Select days and grid points for the heatwave of interest and convert to bool array
da_temp_htw = da_temp.where(da==label, drop=True)
da_HWMId_htw = da_HWMId_year.where(da==label, drop=True)

print(year)
print(da_temp_htw.time.data[0])
print(da_temp_htw.time.data[-1])

1975


IndexError: index 0 is out of bounds for axis 0 with size 0

In [22]:
da_HWMId_year == da_temp

<xarray.DataArray (time: 0, rlat: 412, rlon: 424)> Size: 0B
array([], shape=(0, 412, 424), dtype=bool)
Coordinates:
  * time     (time) object 0B 
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB 21.99 22.03 22.07 ... 66.81 66.75 66.69
    lon      (rlat, rlon) float32 699kB -10.06 -9.964 -9.864 ... 64.76 64.96
    height   float64 8B 2.0
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [31]:
da_temp == da_HWMId_year

<xarray.DataArray (time: 0, rlat: 412, rlon: 424)> Size: 0B
array([], shape=(0, 412, 424), dtype=bool)
Coordinates:
  * time     (time) object 0B 
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB 21.99 22.03 22.07 ... 66.81 66.75 66.69
    lon      (rlat, rlon) float32 699kB -10.06 -9.964 -9.864 ... 64.76 64.96
    height   float64 8B 2.0
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [135]:
da_label

<xarray.DataArray 'label' (time: 92, rlat: 412, rlon: 424)> Size: 129MB
array([[[0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0],
        ...,
        [0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0]],

       [[0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0],
        ...,
        [0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0]],

       ...,

       [[0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0],
        ...,
        [0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0]],

       [[0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0],
        ...,
        [0, 0, ..., 0, 0],
        [0, 0, ..., 0, 0]]], shape=(92, 412, 424))
Coordinates:
  * time     (time) object 736B 1975-06-01 12:00:00 ... 1975-08-31 12:00:00
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB ...
    lon      (rlat, rlon) float32 699kB ...
    height   float64 8B ...
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [133]:
da_temp

<xarray.DataArray 'tasmax' (time: 92, rlat: 412, rlon: 424)> Size: 64MB
array([[[-1.3689566e-01, -1.1055279e-01,  5.0261259e-01, ...,
         -3.8956778e+00, -3.9335933e+00, -3.7604187e+00],
        [-7.5177622e-01, -6.0574961e-01, -2.2930622e-02, ...,
         -3.8993590e+00, -3.9462707e+00, -3.9864011e+00],
        [-1.2202330e+00, -8.2026339e-01, -5.7155132e-01, ...,
         -4.0810790e+00, -4.1084409e+00, -4.1959162e+00],
        ...,
        [-6.4670649e+00, -7.0083618e+00, -8.5161495e+00, ...,
         -7.2833185e+00,  9.5103931e-01,  7.4940348e-01],
        [-6.8214107e+00, -1.0319958e+01, -7.3326960e+00, ...,
         -4.8543863e+00, -5.8383970e+00,  7.6203108e-01],
        [-1.3360717e+00, -8.0615168e+00, -1.2260387e+01, ...,
          3.6239624e-05,  5.9171772e-01,  8.3483315e-01]],

       [[ 1.5273318e+00,  1.5057311e+00,  1.6664491e+00, ...,
         -2.4964476e+00, -2.7174132e+00, -2.8171806e+00],
        [ 4.5847178e-01,  1.1181092e+00,  1.5852604e+00, ...,
         -2.3938534e+00, -2.6057062e+00, -2.9083006e+00],
        [-9.3664646e-02,  9.4893217e-01,  1.6156130e+00, ...,
         -2.5033081e+00, -2.7585688e+00, -3.0148859e+00],
...
         -1.0567431e+01, -9.8140373e+00, -8.9204216e+00],
        [-3.2179008e+00, -3.1655698e+00, -3.9297786e+00, ...,
         -1.0544299e+01, -1.0306255e+01, -9.0109291e+00],
        [-4.6977043e+00, -4.9597163e+00, -4.5871329e+00, ...,
         -1.0162554e+01, -1.0599487e+01, -9.1480513e+00]],

       [[-2.9178524e+00, -3.3751154e+00, -3.9286189e+00, ...,
         -2.6867242e+00, -2.8403440e+00, -2.9850826e+00],
        [-3.0272217e+00, -3.5453000e+00, -3.9788389e+00, ...,
         -2.8320003e+00, -2.9851618e+00, -3.1422727e+00],
        [-3.1286435e+00, -3.5112791e+00, -3.9144711e+00, ...,
         -3.0500610e+00, -3.1345272e+00, -3.4464042e+00],
        ...,
        [-3.4938717e+00, -4.5010433e+00, -5.7117910e+00, ...,
         -1.4258456e+01, -1.3070002e+01, -1.1949280e+01],
        [-4.4610472e+00, -3.8348875e+00, -4.5954037e+00, ...,
         -1.5057702e+01, -1.3660746e+01, -1.2167120e+01],
        [-6.7655878e+00, -6.5632625e+00, -5.0485291e+00, ...,
         -1.5129894e+01, -1.4867753e+01, -1.2714500e+01]]],
      shape=(92, 412, 424), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 736B 1975-06-01T12:00:00 ... 1975-08-31T12...
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB ...
    lon      (rlat, rlon) float32 699kB ...
    height   float64 8B ...
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [130]:
da_temp.where(da==label, drop=False)

<xarray.DataArray 'tasmax' (time: 0, rlat: 412, rlon: 424)> Size: 0B
array([], shape=(0, 412, 424), dtype=float32)
Coordinates:
  * time     (time) object 0B 
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB 21.99 22.03 22.07 ... 66.81 66.75 66.69
    lon      (rlat, rlon) float32 699kB -10.06 -9.964 -9.864 ... 64.76 64.96
    height   float64 8B 2.0
Attributes:
    standard_name:     air_temperature
    long_name:         Daily Maximum Near-Surface Air Temperature
    comment:           daily-maximum near-surface (usually, 2 meter) air temp...
    units:             K
    cell_methods:      time: maximum
    history:           2017-01-06T13:53:00Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [41]:
da_ghs_pop = xr.open_dataset(join("Data/GHS-POP","GHS_POP_1975_2030_Lambert_Conformal.nc"),engine='netcdf4').Band1
da_fpop = xr.open_dataset(join("Data/FPOP","FPOP_SSP5_2020_2100_Lambert_Conformal.nc"),engine='netcdf4').Band1

In [46]:
da_fpop.sel(time=(da_fpop.time.dt.year==2020)).sum()

<xarray.DataArray 'Band1' ()> Size: 4B
array(4.762675e+06, dtype=float32)
Attributes:
    long_name:           GDAL Band Number 1
    grid_mapping:        Lambert_Conformal
    RepresentationType:  ATHEMATIC

In [47]:
da_ghs_pop.sel(time=(da_ghs_pop.time.dt.year==2020)).sum()

<xarray.DataArray 'Band1' ()> Size: 8B
array(4126865.63938711)
Attributes:
    long_name:     GDAL Band Number 1
    grid_mapping:  Lambert_Conformal

In [5]:
# Create sorted list of all files in directory
if read_directory_rcp == None and read_directory_historical == None:
    raise ValueError("Both read_directory_rcp and read_directory_historical are None. At least one must be defined")
elif read_directory_historical == None:
    existing_files = np.sort(glob.glob(join(read_directory_rcp,"*.nc")))
elif read_directory_rcp == None:
    existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc")))
else:
    existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc"))+glob.glob(join(read_directory_rcp,"*.nc")))
# Choose file matching the study period
files_to_load = []
for file in existing_files:
    file_start_year = int(file[-20:-16])
    file_end_year = int(file[-11:-7])
    if (file_start_year >= start_year and file_start_year <= end_year) or (file_end_year >= start_year and file_end_year <= end_year):
        files_to_load.append(str(file))
# Check time consistency
if len(files_to_load)==0:
    raise ValueError("files_to_load is empty. Check directories and time boundaries.")

first_file_start_year = int(files_to_load[0][-20:-16])
if start_year < first_file_start_year :
    raise ValueError(f"Start year {start_year} is before beginning of first file {first_file_start_year}.")
last_file_end_year = int(files_to_load[-1][-11:-7])
if end_year > last_file_end_year :
    raise ValueError(f"End year {end_year} is after ending of last file {last_file_end_year}.")

for i in range(1,len(files_to_load)):
    previous_file_end_year = int(files_to_load[i-1][-11:-7])
    file_start_year = int(files_to_load[i][-20:-16])
    if file_start_year != previous_file_end_year+1 :
        raise ValueError(f"Time is not consistent in files_to_load. Missing year(s) between {previous_file_end_year} and {file_start_year}.")
print(f"Time is consistent between start year {start_year} and end year {end_year}.")

Time is consistent between start year 2000 and end year 2010.


In [6]:
files_to_load

['Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc',
 'Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_20010101-20051231.nc',
 'Data/rcp85/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v2_day_20060101-20101231.nc']

In [19]:
year = 2000
i = 0
found_file=False
file=None
while i<len(files_to_load) and (found_file==False):
    file_start_year = int(files_to_load[i][-20:-16])
    file_end_year = int(files_to_load[i][-11:-7])
    if year>=file_start_year and year<=file_end_year :
        file = files_to_load[i]
        found_file = True
    i+=1
print(found_file,file)

True Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc


In [ ]:
loaded_file = None
old_loaded_file = None
for year in range(1996,2011):
    print(year)
    found_file = False
    i = 0
    while i<len(files_to_load) and (found_file==False):
        file_start_year = int(files_to_load[i][-20:-16])
        file_end_year = int(files_to_load[i][-11:-7])
        if year>=file_start_year and year<=file_end_year :
            loaded_file = files_to_load[i]
            found_file = True
        i+=1
    if found_file:
        if loaded_file != old_loaded_file: # Only load file if change of file
            ds = xr.open_dataset(loaded_file,engine='netcdf4')
            print("New file")
            print(found_file,file)
        old_loaded_file  = loaded_file # Update old_loaded_file value for next iteration
    else:
        raise ValueError(f"File not found for year {year} in following list of files:\n{files_to_load}")


1996
New file
True Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc
1997
1998
1999
2000
2001
New file
True Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc
2002
2003
2004
2005
2006
New file
True Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc
2007
2008
2009
2010
2011


ValueError: File not found for year 2011 in following list of files:
['Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_19960101-20001231.nc', 'Data/historical/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_historical_r1i1p1_GERICS-REMO2015_v2_day_20010101-20051231.nc', 'Data/rcp85/tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v2_day_20060101-20101231.nc']

In [ ]:
GCM, RCM, RCP, version : ('CNRM-CERFACS-CNRM-CM5', 'GERICS-REMO2015', 'rcp85', 'v1')
read_directory_historical : /bdd/CORDEX/output/EUR-11/GERICS/CNRM-CERFACS-CNRM-CM5/historical/r1i1p1/GERICS-REMO2015/v1/day/tasmax/v20170208
read_directory_rcp : /bdd/CORDEX/output/EUR-11/GERICS/CNRM-CERFACS-CNRM-CM5/rcp85/r1i1p1/GERICS-REMO2015/v1/day/tasmax/v20170208
temp_variable : tasmax
start_year_ref : 1975
end_year_ref : 2025


In [ ]:
split_year = 2005

def create_files_list_to_load(read_directory_historical=None,read_directory_rcp=None,start_year=1975,end_year=split_year):
    """Creates the list of files to load, and checks for the time consistency of the created list."""
    # Create sorted list of all files in directory
    if read_directory_rcp == None and read_directory_historical == None:
        raise ValueError("Both read_directory_rcp and read_directory_historical are None. At least one must be defined")
    elif read_directory_historical == None:
        existing_files = np.sort(glob.glob(join(read_directory_rcp,"*.nc")))
    elif read_directory_rcp == None:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc")))
    else:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc"))+glob.glob(join(read_directory_rcp,"*.nc")))
    # Select files matching the study period
    files_to_load = []
    for file in existing_files:
        file_start_year = int(file[-20:-16])
        file_end_year = int(file[-11:-7])
        if (file_start_year >= start_year and file_start_year <= end_year) or (file_end_year >= start_year and file_end_year <= end_year):
            files_to_load.append(str(file))
    # Check time consistency
    if len(files_to_load)==0:
        raise ValueError("files_to_load is empty. Check directories and time boundaries.")

    first_file_start_year = int(files_to_load[0][-20:-16])
    if start_year < first_file_start_year :
        raise ValueError(f"Start year {start_year} is before beginning of first file {first_file_start_year}.")
    last_file_end_year = int(files_to_load[-1][-11:-7])
    if end_year > last_file_end_year :
        raise ValueError(f"End year {end_year} is after ending of last file {last_file_end_year}.")

    for i in range(1,len(files_to_load)):
        previous_file_end_year = int(files_to_load[i-1][-11:-7])
        file_start_year = int(files_to_load[i][-20:-16])
        if file_start_year != previous_file_end_year+1 :
            raise ValueError(f"Time is not consistent in files_to_load. Missing year(s) between {previous_file_end_year} and {file_start_year}.")
    print(f"Time is consistent between start year {start_year} and end year {end_year}.")
    return(files_to_load)

In [ ]:
def compute_seasonal_cycle(read_directory_historical,read_directory_rcp,write_directory,start_year_ref=1975,end_year_ref=2025,temp_variable='tasmax') :
    '''This function computes a seasonal_cycle for each calendar day of the year. 
    By default, the seasonal_cycle is computed over 1975-2025.
    This function can be used with several models and variables.'''

    #Create list of files to load
    if end_year_ref <= split_year:
        files_to_load = create_files_list_to_load(read_directory_historical=read_directory_historical,read_directory_rcp=None,start_year=start_year_ref,end_year=end_year_ref)
    else:
        files_to_load = create_files_list_to_load(read_directory_historical,read_directory_rcp,start_year_ref,end_year_ref)

    # Load multi-file dataset
    #client = Client(n_workers=20, threads_per_worker=2, memory_limit='60GB')
    ds = xr.open_mfdataset(files_to_load, engine='netcdf4',data_vars='all',chunks={'time': 1461})
    try:
        ds = ds.chunk(chunks={'time':len(ds.time),'rlat': 82,'rlon': 84})
    except:
        ds = ds.chunk(chunks={'time':len(ds.time),'x': 82,'y': 84})
    da = getattr(ds, temp_variable)
    # Since the files generally cover several years, have to select sub-period (it may not exactly match the boundaries of loaded files)
    mask = (da.time.dt.year>=start_year_ref) & (da.time.dt.year<=end_year_ref)
    da = da.sel(time=mask)
    del mask

    # Drop Feb 29
    da = da.convert_calendar("noleap")
    # Group using dayofyear and sum to compute mean at the end
    seasonal_cycle = da.groupby(da.time.dt.dayofyear).mean(dim="time")

    # Export data to netcdf file
    seasonal_cycle.to_netcdf(join(write_directory,f"seasonal_cycle_{start_year_ref}_{end_year_ref}.nc"))

    # Clear resources
    da.close()
    seasonal_cycle.close()
    ds.close()
    return

In [ ]:
def compute_distrib_percentile(read_directory_historical,read_directory_rcp,write_directory,start_year_ref=1975,end_year_ref=2025,temp_variable='tasmax',threshold_value=95,distrib_window_size=15,anomaly=True) :
    '''This function computes, for every calendar day, the n-th (n is the threshold_value, default 95) percentile of the corresponding distribution of daily variable. 
    By default, the distribution is computed over 1975-2025.'''

    if distrib_window_size%2==0:
        raise ValueError('distrib_window_size is even. It has to be odd so the window can be centered on the computed day.')

    #Create list of files to load
    if end_year_ref <= split_year: # If all files are in historical directory, ignore rcp directory
        files_to_load = create_files_list_to_load(read_directory_historical=read_directory_historical,read_directory_rcp=None,start_year=start_year_ref,end_year=end_year_ref)
    else:
        files_to_load = create_files_list_to_load(read_directory_historical,read_directory_rcp,start_year_ref,end_year_ref)

    # Load multi-file dataset
    #client = Client(n_workers=20, threads_per_worker=2, memory_limit='60GB')
    ds = xr.open_mfdataset(files_to_load, engine='netcdf4',data_vars='all',chunks={'time': 1461})#,parallel=True)
    try:
        ds = ds.chunk(chunks={'time':len(ds.time),'rlat': 82,'rlon': 84})
    except:
        ds = ds.chunk(chunks={'time':len(ds.time),'x': 82,'y': 84})
    da = getattr(ds, temp_variable)
    # Since files generally cover several years, have to select sub-period (it may not exactly match the boundaries of loaded files)
    mask = (da.time.dt.year>=start_year_ref) & (da.time.dt.year<=end_year_ref)
    da = da.sel(time=mask)
    del mask
    
    # Load seasonal_cycle file to create output data structure and compute anomaly
    seasonal_cycle = xr.open_dataarray(join(write_directory,f"seasonal_cycle_{start_year_ref}_{end_year_ref}.nc"), engine='netcdf4')

    # Drop 29 Feb
    da = da.convert_calendar("noleap")

    # Create threshold table by copying seasonal_cycle table, values will be updated later
    threshold = seasonal_cycle.copy()

    if anomaly :
        for year in tqdm(range(len(da.time)//365)) : # Iterate over the number of years
            da[year*365:(year+1)*365,:,:] = da[year*365:(year+1)*365,:,:] - seasonal_cycle.data # Compute anomaly
    else :
        seasonal_cycle.close()
    
    for day in tqdm(range(1,366)) : # Calendar days ranging from 1 to 365 (no leap years)
        if day - (distrib_window_size//2) <= 0  : # day is at the beginning of January, window overlapping with December
            distrib_start_bound = 365 + day - (distrib_window_size//2) # start bound of the distribution window
            distrib_end_bound = day + (distrib_window_size//2) # end bound of the distribution window
            mask = (da.time.dt.dayofyear >= distrib_start_bound) + (da.time.dt.dayofyear <= distrib_end_bound) # Create a boolean mask for days between distrib_start_bound and distrib_end_bound
        elif day + (distrib_window_size//2) > 365 : # day is at the end of December , window overlapping with January
            distrib_start_bound = day - (distrib_window_size//2)
            distrib_end_bound = day + (distrib_window_size//2) - 365
            mask = (da.time.dt.dayofyear >= distrib_start_bound) + (da.time.dt.dayofyear <= distrib_end_bound) # Create a boolean mask for days between distrib_start_bound and distrib_end_bound
        else :
            distrib_start_bound = day - (distrib_window_size//2)
            distrib_end_bound = day + (distrib_window_size//2)
            mask = (da.time.dt.dayofyear >= distrib_start_bound) & (da.time.dt.dayofyear <= distrib_end_bound) # Create a boolean mask for days between distrib_start_bound and distrib_end_bound
        # Apply the mask to select the corresponding days and compute percentile, take day-1 because day ranges from 1 to 365 but python indexing ranges from 0 to 364
        if day == 1:
            print(np.shape(da.sel(time=mask).data))
        threshold.data[day-1,:,:] = np.percentile(da.sel(time=mask).data,threshold_value,axis=0)

    # Export data to netcdf file
    threshold.to_netcdf(join(write_directory,f"distrib_threshold_{threshold_value}.nc"))

    # Clear resources
    da.close()
    threshold.close()
    if anomaly :
        seasonal_cycle.close()
    ds.close()
    return

In [11]:
write_directory = "Data/output"
compute_seasonal_cycle(read_directory_historical,read_directory_rcp,write_directory,start_year_ref=2000,end_year_ref=2010,temp_variable='tas')

Time is consistent between start year 2000 and end year 2010.


In [14]:
compute_distrib_percentile(read_directory_historical,read_directory_rcp,write_directory,start_year_ref=2000,end_year_ref=2010,temp_variable='tas',threshold_value=95,distrib_window_size=15,anomaly=True)

Time is consistent between start year 2000 and end year 2010.


  0%|          | 0/365 [00:00<?, ?it/s]

(165, 412, 424)


  1%|▏         | 5/365 [01:42<2:03:21, 20.56s/it]


KeyboardInterrupt: 

In [7]:
ds = xr.open_mfdataset(files_to_load, engine='netcdf4',data_vars='all')#chunks={'time': 1461},data_vars='all',parallel=True)
ds = ds.chunk(chunks={'time':len(ds.time),'rlat': 82,'rlon': 84})
da = getattr(ds, 'tas')

In [8]:
da

<xarray.DataArray 'tas' (time: 5479, rlat: 412, rlon: 424)> Size: 4GB
dask.array<rechunk-merge, shape=(5479, 412, 424), dtype=float32, chunksize=(5479, 82, 84), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 44kB 1996-01-01T12:00:00 ... 2010-12-31T12...
  * rlat     (rlat) float64 3kB -23.38 -23.27 -23.16 ... 21.61 21.72 21.84
  * rlon     (rlon) float64 3kB -28.38 -28.27 -28.16 ... 17.93 18.04 18.16
    lat      (rlat, rlon) float32 699kB dask.array<chunksize=(82, 84), meta=np.ndarray>
    lon      (rlat, rlon) float32 699kB dask.array<chunksize=(82, 84), meta=np.ndarray>
    height   float64 8B 2.0
Attributes:
    standard_name:     air_temperature
    long_name:         Near-Surface Air Temperature
    comment:           daily-mean near-surface (usually, 2 meter) air tempera...
    units:             K
    original_name:     tas
    cell_methods:      time: mean
    history:           2020-06-13T04:37:48Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [11]:
mask = (da.time.dt.year>=start_year) & (da.time.dt.year<=end_year)

In [12]:
# Since files generally cover several years, have to select sub-period (it may not exactly match the boundaries of loaded files)
da = da.sel(time=mask)

In [1]:
da

NameError: name 'da' is not defined

In [ ]:
def find_correct_year_file(read_directory,start_year,interval) :
    onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
    pattern = re.compile(f"{start_year}0101-{start_year+interval-1}123[0-1].nc$")
    for file in onlyfiles :
        match = pattern.search(file)
        if match is not None :
            break
    return file

In [128]:
def check_time_consistency(read_directory,interval=5,start_year=1951,end_year=2005) :
    """Checks that the time intervals are consistent with pre-defined parameters. 
    Returns True if files present in read_directory are consistent, False otherwise"""
    if (end_year-start_year+1)%interval != 0 :
        raise ValueError(f"Incorrect input. Inconsistency between start_year ({start_year}), end_year ({end_year}), and interval ({interval}).")
    onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
    for year in range(start_year,end_year,interval) :
        pattern = re.compile(f"{year}0101-{year+interval-1}123[0-1].nc$")
        flag = np.array([not(pattern.search(file) is None) for file in onlyfiles]).any() # flag is True if there is a match for one of the files, False otherwise
        if not flag :
            break
    return flag

In [129]:
check_time_consistency(os.getcwd(),5,2006,2100)

np.True_

In [ ]:
#os.chdir("../../Code")
os.getcwd()

['tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20810101-20851231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20360101-20401231.nc',
 '.git',
 'CORDEX.json',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20310101-20351231.nc',
 'CORDEX.ipynb',
 'export.csv',
 'catalogue.ipynb',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20860101-20901231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20610101-20651231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20460101-20501231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20760101-20801231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20560101-20601231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20260101-20301231.nc',
 'CORDEX.csv.gz',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO

In [130]:
read_directory=os.getcwd()
write_directory=os.getcwd()
start_year_clim=2006
end_year_clim=2100
interval=5
temp_variable='tas'
smooth_span=15

In [134]:
onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
year_list = range(start_year_clim,end_year_clim,interval)

# Load files in a dictionary
for year in year_list :
    pattern = re.compile(f"{year}0101-{year+interval-1}123[0-1].nc$") #chose file from start year and interval
    ds_dict[year] = xr.open_dataset(join(read_directory,find_correct_year_file(read_directory,year,interval)), engine="netcdf4") 

# Initialize output data array with the first file
da = getattr(ds_dict[start_year_clim], temp_variable)

# Drop Feb 29
not_feb29 = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
da = da.sel(time=not_feb29)
# Create synthetic dayofyear: reset every year to 1–365
# Do this manually to break reliance on the original calendar
times = da.time
years = times.dt.year.values
months = times.dt.month.values
days = times.dt.day.values
# Create a structured datetime array to use as keys
# Create dummy dates all in a non-leap year (e.g., 2001)
uniform_dates = pd.to_datetime([f"2001-{month:02d}-{day:02d}" for month, day in zip(months, days)])
# Extract synthetic dayofyear 
dayofyear_365 = xr.DataArray(uniform_dates.dayofyear,coords={"time": times},dims="time")
# Group using this clean synthetic dayofyear and summing to compute mean at the end
climatology = da.groupby(dayofyear_365).sum("time")

for year in tqdm(year_list[1:]) : #iterate over files, except first one which has already been used in initialization
    da = getattr(ds_dict[start_year_clim], temp_variable)
    not_feb29 = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    da = da.sel(time=not_feb29)
    times = da.time
    years = times.dt.year.values
    months = times.dt.month.values
    days = times.dt.day.values
    uniform_dates = pd.to_datetime([f"2001-{month:02d}-{day:02d}" for month, day in zip(months, days)])
    dayofyear_365 = xr.DataArray(uniform_dates.dayofyear,coords={"time": times},dims="time")
    da = da.groupby(dayofyear_365).sum("time")
    climatology += da
    # Rename the dimension
climatology /= end_year_clim - start_year_clim +1
climatology = climatology.rename({"group": "dayofyear"})

100%|██████████| 18/18 [01:07<00:00,  3.74s/it]


In [144]:
extended_temp=np.zeros((365*3,np.shape(climatology.data)[1],np.shape(climatology.data)[2]))

In [ ]:
extended_temp[0:365,:,:]=climatology.data[:,:,:]
extended_temp[365:730,:,:]=climatology.data[:,:,:]
extended_temp[730:,:,:]=climatology.data[:,:,:]

In [160]:
# Smoothing
print("Smoothing...")
for i in tqdm(range(365,730)):
    val_table=np.array(np.zeros((2*smooth_span+1,np.shape(climatology.data)[1],np.shape(climatology.data)[2])))
    for j in range(-smooth_span,smooth_span+1,1):
        val_table[j]=extended_temp[i+j,:,:]
    climatology[i-365,:,:] = np.mean(val_table[:],axis=0)

Smoothing...


100%|██████████| 365/365 [00:05<00:00, 66.13it/s]


In [157]:
climatology.to_netcdf(os.path.join(write_directory,f"{temp_variable}_climatology_{start_year_clim}_{end_year_clim}.nc"))

In [161]:
climatology.close()

In [31]:
import numpy as np
import sys
from tqdm import tqdm # Create a user-friendly feedback for loops while script is running
import os
from os.path import join
import time 

In [ ]:
combination = tuple(('CNRM-CERFACS-CNRM-CM5','GERICS-REMO2015','rcp85','v2'))
read_directory_historical = "/home/user/These/cordex_htws_cc3d/Data/historical"#sys.argv[5]
read_directory_rcp = "/home/user/These/cordex_htws_cc3d/Data/rcp85"#sys.argv[6]
write_directory = "/home/user/These/cordex_htws_cc3d/Data/output"
os.makedirs(write_directory,exist_ok=True)

start_year = 1975
end_year = 2099
start_year_ref = 1975
end_year_ref = 2025
temp_variable = 'tas' # Will need to change to tasmax
threshold_value = 95
distrib_window_size = 15
anomaly = True
relative_threshold = True
nb_days = 4
connectivity = 26

In [ ]:
import numpy as np
import numpy.ma as ma #use masked array
import xarray as xr
import netCDF4 as nc #load and write netcdf data
from datetime import date, timedelta, datetime #create file history with creation date
from tqdm import tqdm #create a user-friendly feedback while script is running
from os import listdir
from os.path import isfile, join
import re #Use RegEx 
import pandas as pd #handle dataframes
import cc3d #connected components patterns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as clrs
from scipy import stats
from scipy import signal
import json
import glob
import dask
from dask.distributed import Client

# Split year is the last year of the historical period. In CORDEX runs based on CMIP5 experiment, it is 2005.
# Have to change value for a different version of CORDEX. 
split_year = 2005
def create_files_list_to_load(read_directory_historical=None,read_directory_rcp=None,start_year=1975,end_year=split_year):
    """Creates the list of files to load, and checks for the time consistency of the created list."""
    # Create sorted list of all files in directory
    if read_directory_rcp == None and read_directory_historical == None:
        raise ValueError("Both read_directory_rcp and read_directory_historical are None. At least one must be defined")
    elif read_directory_historical == None:
        existing_files = np.sort(glob.glob(join(read_directory_rcp,"*.nc")))
    elif read_directory_rcp == None:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc")))
    else:
        existing_files = np.sort(glob.glob(join(read_directory_historical,"*.nc"))+glob.glob(join(read_directory_rcp,"*.nc")))
    # Select files matching the study period
    files_to_load = []
    for file in existing_files:
        file_start_year = int(file[-20:-16])
        file_end_year = int(file[-11:-7])
        if (file_start_year >= start_year and file_start_year <= end_year) or (file_end_year >= start_year and file_end_year <= end_year):
            files_to_load.append(str(file))
    # Check time consistency
    if len(files_to_load)==0:
        raise ValueError("files_to_load is empty. Check directories and time boundaries.")

    first_file_start_year = int(files_to_load[0][-20:-16])
    if start_year < first_file_start_year :
        raise ValueError(f"Start year {start_year} is before beginning of first file {first_file_start_year}.")
    last_file_end_year = int(files_to_load[-1][-11:-7])
    if end_year > last_file_end_year :
        raise ValueError(f"End year {end_year} is after ending of last file {last_file_end_year}.")

    for i in range(1,len(files_to_load)):
        previous_file_end_year = int(files_to_load[i-1][-11:-7])
        file_start_year = int(files_to_load[i][-20:-16])
        if file_start_year != previous_file_end_year+1 :
            raise ValueError(f"Time is not consistent in files_to_load. Missing year(s) between {previous_file_end_year} and {file_start_year}.")
    print(f"Time is consistent between start year {start_year} and end year {end_year}.")
    return(files_to_load)

In [54]:
ref_period_offset=0.72
running_mean_window_size=20
regional_warming_levels_list=[2.1,2.6,4.0,5.1]

if running_mean_window_size % 2 != 0:
    raise ValueError(f"running_mean_window_size must be an even integer, found {running_mean_window_size}")

# Create list of files to load
if end_year <= split_year: # If all files are in historical directory, ignore rcp directory
    files_to_load = create_files_list_to_load(read_directory_historical=read_directory_historical,read_directory_rcp=None,start_year=start_year,end_year=end_year)
elif start_year > split_year: # If all files are in rcp directory, ignore historical directory
    files_to_load = create_files_list_to_load(read_directory_historical=None,read_directory_rcp=read_directory_rcp,start_year=start_year,end_year=end_year)
else:
    files_to_load = create_files_list_to_load(read_directory_historical,read_directory_rcp,start_year,end_year)

for i in range(len(files_to_load)): # RWL are computed on average temperature ('tas'), regardless of the chosen temp_variable
    tas_file = files_to_load[i].replace(temp_variable,'tas')
    files_to_load[i] = tas_file

# Load multi-file dataset
#client = Client(n_workers=20, threads_per_worker=2, memory_limit='60GB')
ds = xr.open_mfdataset(files_to_load, engine='netcdf4',chunks={'time': 1461},data_vars='all')
# Variable is necessarily 'tas' for the computation of warming levels
da = ds.tas
# Since files generally cover several years, have to select sub-period (it may not exactly match the boundaries of loaded files)
mask = (da.time.dt.year>=start_year) & (da.time.dt.year<=end_year)
da = da.sel(time=mask)
del mask
da = da.groupby(da.time.dt.year).mean() # Compute annual mean
da = da.chunk(chunks={'year':len(da.year)})
# Create reference period (default 1986-2005) average
mask = (da.year >= start_year_ref)&(da.year <= end_year_ref)
da_ref = da.sel(year=mask)
da_ref = da_ref.mean(dim="year") # Average over time, 1 time step remaining

da = da.rolling(year=running_mean_window_size, center=True).mean() # Compute running mean



Time is consistent between start year 1971 and end year 2099.


In [55]:
# Compute latitude-weighted mean to obtain 1D-array of annual mean
weights = np.cos(np.deg2rad(da.lat))
weights.name = "weights"
da_weighted = da.weighted(weights)
da_ref = da_ref.weighted(weights)

try : # Have to take into account the fact that grid_mapping is not always the same in CORDEX outputs
    da_weighted = da_weighted.mean(("rlat","rlon")) # 1D-array of annual values
    da_ref = da_ref.mean(("rlat","rlon")) # one single value
except :
    da_weighted = da_weighted.mean(("x","y")) # 1D-array of annual values
    da_ref = da_ref.mean(("x","y")) # one single value
    
da_weighted = da_weighted - da_ref # 1D-array of annual values reprenting the european warming since the reference period (default 1986-2005)


In [58]:
da_weighted = da_weighted.compute()

In [60]:
da_weighted

<xarray.DataArray 'tas' (year: 129)> Size: 516B
array([        nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
       -0.4133606 , -0.4079895 , -0.42074585, -0.42303467, -0.40353394,
       -0.39401245, -0.35910034, -0.33151245, -0.3163147 , -0.30838013,
       -0.25463867, -0.21627808, -0.17819214, -0.1506958 , -0.11813354,
       -0.06311035, -0.04397583, -0.00796509,  0.01376343,  0.03652954,
        0.05230713,  0.06130981,  0.13253784,  0.17092896,  0.20263672,
        0.26431274,  0.27764893,  0.28860474,  0.2998047 ,  0.34490967,
        0.35653687,  0.3534546 ,  0.39035034,  0.4086609 ,  0.43624878,
        0.4369812 ,  0.43075562,  0.43936157,  0.46914673,  0.49346924,
        0.5315552 ,  0.56134033,  0.56903076,  0.59869385,  0.59350586,
        0.6000061 ,  0.6446228 ,  0.69592285,  0.7345276 ,  0.75427246,
        0.8050842 ,  0.84402466,  0.8680725 ,  0.89624023,  0.9069824 ,
        0.9364929 ,  0.9777832 ,  1.023529  ,  1.0396118 ,  1.0758667 ,
        1.1034851 ,  1.1399231 ,  1.1848755 ,  1.2323914 ,  1.3036499 ,
        1.3492737 ,  1.402832  ,  1.4622498 ,  1.5268555 ,  1.5688782 ,
        1.5816345 ,  1.6257629 ,  1.6773376 ,  1.7428589 ,  1.8082886 ,
        1.8755188 ,  1.924469  ,  1.970581  ,  2.0375671 ,  2.0941162 ,
        2.1476746 ,  2.2097168 ,  2.2554626 ,  2.2904663 ,  2.3261719 ,
        2.4020386 ,  2.4510193 ,  2.490448  ,  2.523529  ,  2.575531  ,
        2.6539001 ,  2.7236633 ,  2.7649841 ,  2.8065796 ,  2.8609314 ,
        2.900299  ,  2.9634705 ,  3.0107727 ,  3.0708618 ,  3.1081543 ,
        3.13385   ,  3.192505  ,  3.242279  ,  3.3310852 ,  3.3959045 ,
        3.4261475 ,  3.4562378 ,  3.5073547 ,  3.560852  ,  3.6314392 ,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan], dtype=float32)
Coordinates:
  * year     (year) int64 1kB 1971 1972 1973 1974 1975 ... 2096 2097 2098 2099
    height   float64 8B 2.0
Attributes:
    standard_name:     air_temperature
    long_name:         Near-Surface Air Temperature
    comment:           daily-mean near-surface (usually, 2 meter) air tempera...
    units:             K
    original_name:     tas
    cell_methods:      time: mean
    history:           2020-06-13T04:43:05Z altered by CMOR: Treated scalar d...
    associated_files:  gridspecFile: gridspec_atmos_fx_GERICS-REMO2015_histor...
    grid_mapping:      rotated_latitude_longitude

In [59]:
# Create dictionary to hold the data of
warming_levels_dict = {}
for level in regional_warming_levels_list :
    level_diff = level - ref_period_offset # Since reference period is not 1850-1900, have to introduce offset (default is 0.61°C for 1986-2005 reference period)
    # Find years warmer than level and get the first matching year
    bool_array = (da_weighted - level_diff > 0)
    if not bool_array.any() : # If the corresponding level has not been reached by the model
        warming_levels_dict[level] = None
    else :
        central_year = int(bool_array.idxmax())
        warming_levels_dict[level] = {"start_year" : int(central_year - running_mean_window_size/2), "end_year" : int(central_year + (running_mean_window_size/2 - 1))}

print(warming_levels_dict)
#with open(join(write_directory,'regional_warming_levels.json'), 'w') as fp:
#    json.dump(warming_levels_dict, fp)


{2.1: {'start_year': 2037, 'end_year': 2056}, 2.6: {'start_year': 2047, 'end_year': 2066}, 4.0: {'start_year': 2074, 'end_year': 2093}, 5.1: None}
